<a href="https://colab.research.google.com/github/sajalgoyal007/Games/blob/main/Bank_Management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bank Account  
Deposit Money   
Withdraw Money    
Details   
Update Details    
Delete the account    
To save the data we are using the JSON file

In [32]:
!pip install streamlit pyngrok pandas -q
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹npm notice
npm notice New major version of npm available! 10.8.2 -> 11.16.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.16.0
npm notice To update run: npm install -g npm@11.16.0
npm notice
⠹

In [33]:
import json

with open("data.json", "w") as f:
    json.dump([], f, indent=4)

print("Database Created")

Database Created


In [35]:
%%writefile app.py


import streamlit as st
import json
import random
import string
from pathlib import Path
from datetime import datetime
import pandas as pd

DATA_FILE = "data.json"

# ---------------- DATABASE ----------------

def load_data():
    if Path(DATA_FILE).exists():
        with open(DATA_FILE, "r") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return [] # Handle empty or malformed JSON
    return []

def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=4)

# ---------------- ACCOUNT GENERATOR ----------------

def generate_account():
    while True:
        account = ''.join(
            random.choices(
                string.ascii_uppercase + string.digits,
                k=8
            )
        )

        exists = any(
            user["accountNo"] == account
            for user in load_data()
        )

        if not exists:
            return account

# ---------------- FIND USER ----------------

def find_user(account, pin):
    data = load_data()

    for user in data:
        if (
            user["accountNo"] == account
            and str(user["pin"]) == str(pin)
        ):
            return user

    return None

# ---------------- APP ----------------

st.set_page_config(
    page_title="Bank Management System",
    page_icon="🏦",
    layout="wide"
)

st.title("🏦 Bank Management System")

menu = st.sidebar.radio(
    "Menu",
    [
        "Create Account",
        "Deposit",
        "Withdraw",
        "Show Details",
        "Update Details",
        "Delete Account",
        "Database"
    ]
)

# ================= CREATE =================

if menu == "Create Account":

    st.header("Create New Account")

    with st.form("create"):

        name = st.text_input("Name")
        age = st.number_input(
            "Age",
            min_value=1,
            step=1
        )

        email = st.text_input("Email")

        pin = st.text_input(
            "4 Digit PIN",
            type="password"
        )

        submit = st.form_submit_button(
            "Create Account"
        )

        if submit:

            if age < 18:
                st.error("Age must be 18+")
                st.stop()

            if (
                len(pin) != 4
                or not pin.isdigit()
            ):
                st.error("PIN must be 4 digits")
                st.stop()

            account = generate_account()

            user = {
                "name": name,
                "age": age,
                "email": email,
                "pin": int(pin),
                "accountNo": account,
                "balance": 0,
                "transactions": []
            }

            data = load_data()
            data.append(user)
            save_data(data)

            st.success("Account Created")
            st.code(account)

# ================= DEPOSIT =================

elif menu == "Deposit":

    st.header("Deposit Money")

    acc = st.text_input("Account Number")
    pin = st.text_input(
        "PIN",
        type="password"
    )

    amount = st.number_input(
        "Amount",
        min_value=1
    )

    if st.button("Deposit"):

        data = load_data()

        for user in data:

            if (
                user["accountNo"] == acc
                and str(user["pin"]) == pin
            ):

                user["balance"] += amount

                user["transactions"].append({
                    "type": "Deposit",
                    "amount": amount,
                    "date": str(datetime.now())
                })

                save_data(data)

                st.success("Deposit Successful")
                st.metric(
                    "Balance",
                    f"₹{user['balance']}"
                )
                st.stop()

        st.error("Invalid Credentials")

# ================= WITHDRAW =================

elif menu == "Withdraw":

    st.header("Withdraw Money")

    acc = st.text_input("Account Number")
    pin = st.text_input(
        "PIN",
        type="password"
    )

    amount = st.number_input(
        "Amount",
        min_value=1
    )

    if st.button("Withdraw"):

        data = load_data()

        for user in data:

            if (
                user["accountNo"] == acc
                and str(user["pin"]) == pin
            ):

                if user["balance"] < amount:
                    st.error(
                        "Insufficient Balance"
                    )
                    st.stop()

                user["balance"] -= amount

                user["transactions"].append({
                    "type": "Withdraw",
                    "amount": amount,
                    "date": str(datetime.now())
                })

                save_data(data)

                st.success("Withdrawal Successful")
                st.metric(
                    "Balance",
                    f"₹{user['balance']}"
                )
                st.stop()

        st.error("Invalid Credentials")

# ================= SHOW DETAILS =================

elif menu == "Show Details":

    st.header("Account Details")

    acc = st.text_input("Account Number")
    pin = st.text_input(
        "PIN",
        type="password"
    )

    if st.button("Show"):

        user = find_user(acc, pin)

        if user:

            st.metric(
                "Current Balance",
                f"₹{user['balance']}"
            )

            st.json({
                "Name": user["name"],
                "Age": user["age"],
                "Email": user["email"],
                "Account": user["accountNo"]
            })

            if user["transactions"]:

                st.subheader(
                    "Transaction History"
                )

                st.dataframe(
                    pd.DataFrame(
                        user["transactions"]
                    )
                )

        else:
            st.error("Invalid Credentials")

# ================= UPDATE =================

elif menu == "Update Details":

    st.header("Update Details")

    acc = st.text_input("Account Number")
    pin = st.text_input(
        "PIN",
        type="password"
    )

    new_name = st.text_input("New Name")
    new_email = st.text_input("New Email")

    if st.button("Update"):

        data = load_data()

        for user in data:

            if (
                user["accountNo"] == acc
                and str(user["pin"]) == pin
            ):

                if new_name:
                    user["name"] = new_name

                if new_email:
                    user["email"] = new_email

                save_data(data)

                st.success(
                    "Details Updated"
                )

                st.stop()

        st.error("Invalid Credentials")

# ================= DELETE =================

elif menu == "Delete Account":

    st.header("Delete Account")

    acc = st.text_input("Account Number")
    pin = st.text_input(
        "PIN",
        type="password"
    )

    if st.button("Delete"):

        data = load_data()

        new_data = [
            user
            for user in data
            if not (
                user["accountNo"] == acc
                and str(user["pin"]) == pin
            )
        ]

        if len(new_data) != len(data):

            save_data(new_data)

            st.success(
                "Account Deleted"
            )

        else:
            st.error(
                "Invalid Credentials"
            )

# ================= DATABASE =================

elif menu == "Database":

    st.header("Database")

    data = load_data()

    if data:

        st.dataframe(data)

        with open(
            DATA_FILE,
            "rb"
        ) as file:

            st.download_button(
                "Download Database",
                file,
                file_name="data.json"
            )

    else:
        st.info("No Records Found")

Overwriting app.py


In [37]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!lt --port 8501

your url is: https://honest-bottles-show.loca.lt


In [ ]:
!curl ipv4.icanhazip.com